# Daymet → Hugging Face Pixel Extraction

GitHub-ready copy of the working Colab extraction notebook.

**Data and credentials are intentionally not stored in this notebook.**
Google Drive paths refer to the Colab-mounted project; Hugging Face authentication
is performed interactively with `login()`.


## Downloading libraries


In [ ]:
!pip -q install pystac-client planetary-computer

In [ ]:
!pip -q install -U huggingface_hub hf_xet

In [ ]:
import pystac_client
import planetary_computer

print("Packages installed successfully")

In [ ]:
!pip -q install zarr fsspec adlfs

## Mounting drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os

print("\nMyDrive exists:", os.path.exists("/content/drive/MyDrive"))

print("\nTop-level MyDrive folders:")
for x in os.listdir("/content/drive/MyDrive"):
    print(" ", x)

## Connecting to hugging face

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import whoami

info = whoami()

print("Logged in as:", info["name"])

In [ ]:
from huggingface_hub import whoami
print(whoami()["name"])

In [ ]:
from huggingface_hub import HfApi, whoami

USERNAME = whoami()["name"]
REPO_NAME = "daymet-camels-pixel-1995-2020"

api = HfApi()

repo_id = api.create_repo(
    repo_id=f"{USERNAME}/{REPO_NAME}",
    repo_type="dataset",
    private=True,
    exist_ok=True
)

print("Repository created/confirmed:")
print(repo_id)

In [ ]:
from huggingface_hub import HfApi

HF_REPO = "vidushibhar24/daymet-camels-pixel-1995-2020"

api = HfApi()

files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type="dataset"
)

chunks = sorted(
    [f for f in files if f.startswith("chunks/chunk_") and f.endswith(".npz")]
)

print("=" * 70)
print("HUGGING FACE CHUNK VERIFICATION")
print("=" * 70)

print("Total chunks on Hugging Face:", len(chunks))

if chunks:
    print("First:", chunks[0])
    print("Last: ", chunks[-1])

expected = {
    f"chunks/chunk_{i:03d}.npz"
    for i in range(1, 99)
}

actual = set(chunks)

missing = sorted(expected - actual)
extra = sorted(actual - expected)

print("\nExpected chunks: 001–098")
print("Missing:", missing if missing else "NONE")
print("Unexpected:", extra if extra else "NONE")

if actual == expected:
    print("\n✓✓✓ ALL 98 CHUNKS VERIFIED ✓✓✓")
else:
    print("\n⚠ Repository does not yet contain exactly chunks 001–098.")

## Actual Extraction

In [ ]:
# ================================================================
# DAYMET → HUGGING FACE DIRECT EXTRACTION
# 1995–2020 | 483,042 pixels | 242 chunks
#
# Existing HF chunks are automatically skipped.
# Current completed chunks: 001–098
# Next chunk: 099
#
# Each NPZ contains ONLY:
#   tmin
#   tmax
#   prcp
#   pet
#
# Metadata remains in the Drive pixel index / metadata files.
# ================================================================

# ---------------------------------------------------------------
# 1. INSTALL PACKAGES
# ---------------------------------------------------------------

!pip -q install pystac-client planetary-computer zarr adlfs huggingface_hub hf_xet


# ---------------------------------------------------------------
# 2. IMPORTS
# ---------------------------------------------------------------

import os
import gc
import time
import shutil
import warnings
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import xarray as xr

import pystac_client
import planetary_computer
import planetary_computer.sas as pc_sas

from huggingface_hub import HfApi, login


warnings.filterwarnings("ignore")


# ---------------------------------------------------------------
# 3. MOUNT GOOGLE DRIVE
# ---------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")


# ---------------------------------------------------------------
# 4. CONFIGURATION
# ---------------------------------------------------------------

ROOT = "/content/drive/MyDrive/whiplash-project"
DATA = f"{ROOT}/data"

PIXEL_FILE = (
    f"{DATA}/daymet_pixel_index_629_basins.csv"
)

METADATA_DIR = (
    f"{DATA}/daymet_chunks_1995_2020"
)

HF_REPO = (
    "vidushibhar24/daymet-camels-pixel-1995-2020"
)

HF_REPO_TYPE = "dataset"

TEMP_DIR = "/content/daymet_hf_temp"

START = "1995-01-01"
END = "2020-12-30"

CELL_CHUNK = 2000

N_CELLS_EXPECTED = 483042
N_TIME_EXPECTED = 9490

MAX_DAYMET_RETRIES = 5

os.makedirs(TEMP_DIR, exist_ok=True)


print("=" * 70)
print("DAYMET → HUGGING FACE DIRECT EXTRACTION")
print("=" * 70)
print(f"Period: {START} to {END}")
print(f"Cells/chunk: {CELL_CHUNK}")
print(f"HF repository: {HF_REPO}")
print(f"Temporary directory: {TEMP_DIR}")
print("=" * 70)


# ---------------------------------------------------------------
# 5. LOAD PIXEL INDEX
# ---------------------------------------------------------------

print("\nLoading pixel index...")

pixels = pd.read_csv(
    PIXEL_FILE
)

print(
    f"Pixel index rows: {len(pixels)}"
)

print(
    "Columns:",
    list(pixels.columns)
)


required_columns = [
    "basin_id",
    "y_index",
    "x_index",
    "daymet_y",
    "daymet_x",
]


missing_columns = [
    c for c in required_columns
    if c not in pixels.columns
]

if missing_columns:

    raise RuntimeError(
        f"Missing required pixel-index columns: "
        f"{missing_columns}"
    )


ncell = len(pixels)

if ncell != N_CELLS_EXPECTED:

    raise RuntimeError(
        f"Expected {N_CELLS_EXPECTED} cells, "
        f"found {ncell}"
    )


y_indices = (
    pixels["y_index"]
    .to_numpy(dtype=np.int64)
)

x_indices = (
    pixels["x_index"]
    .to_numpy(dtype=np.int64)
)


pixel_basin_ids = (
    pixels["basin_id"]
    .astype(str)
    .to_numpy()
)


print(
    f"Total cells: {ncell}"
)

print(
    f"Cells/chunk: {CELL_CHUNK}"
)


TOTAL_CHUNKS = int(
    np.ceil(ncell / CELL_CHUNK)
)


print(
    f"Total chunks: {TOTAL_CHUNKS}"
)


if TOTAL_CHUNKS != 242:

    raise RuntimeError(
        f"Expected 242 chunks, "
        f"got {TOTAL_CHUNKS}"
    )


# ---------------------------------------------------------------
# 6. LOAD METADATA
# ---------------------------------------------------------------

print("\nLoading metadata...")


time_file = (
    f"{METADATA_DIR}/time.npy"
)

doy_file = (
    f"{METADATA_DIR}/doy.npy"
)

lat_file = (
    f"{METADATA_DIR}/lat.npy"
)

lon_file = (
    f"{METADATA_DIR}/lon.npy"
)

basin_ids_file = (
    f"{METADATA_DIR}/basin_ids.npy"
)


for f in [
    time_file,
    doy_file,
    lat_file,
    lon_file,
    basin_ids_file,
]:

    if not os.path.exists(f):

        raise FileNotFoundError(
            f"Required metadata file not found:\n{f}"
        )


dates = np.load(
    time_file,
    allow_pickle=True
)

doy = np.load(
    doy_file
)

lat = np.load(
    lat_file
)

lon = np.load(
    lon_file
)

basin_ids = np.load(
    basin_ids_file,
    allow_pickle=True
)


print(
    f"Time steps: {len(dates)}"
)

print(
    f"Latitude values: {len(lat)}"
)

print(
    f"Longitude values: {len(lon)}"
)

print(
    f"Basin IDs: {len(basin_ids)}"
)


# ---------------------------------------------------------------
# Validate metadata
# ---------------------------------------------------------------

if len(dates) != N_TIME_EXPECTED:

    raise RuntimeError(
        f"Expected {N_TIME_EXPECTED} dates, "
        f"found {len(dates)}"
    )


if len(doy) != N_TIME_EXPECTED:

    raise RuntimeError(
        f"Expected {N_TIME_EXPECTED} DOY values, "
        f"found {len(doy)}"
    )


if len(lat) != ncell:

    raise RuntimeError(
        f"Latitude length {len(lat)} "
        f"does not match cells {ncell}"
    )


if len(lon) != ncell:

    raise RuntimeError(
        f"Longitude length {len(lon)} "
        f"does not match cells {ncell}"
    )


print(
    "Basin-level metadata contains "
    f"{len(basin_ids)} basin IDs."
)

print(
    "Pixel-level basin IDs are stored in the CSV."
)

print(
    "Metadata validation: OK"
)


# ---------------------------------------------------------------
# 7. HARGREAVES-SAMANI PET
# ---------------------------------------------------------------

def hargreaves_samani_pet(
    tmin_arr,
    tmax_arr,
    lat_deg,
    doy_arr
):

    """
    Calculate daily Hargreaves-Samani PET.

    Inputs:
        tmin_arr : (time, cell)
        tmax_arr : (time, cell)
        lat_deg  : (cell,)
        doy_arr  : (time,)

    Output:
        PET in mm/day
    """

    tmin_arr = np.asarray(
        tmin_arr,
        dtype=np.float32
    )

    tmax_arr = np.asarray(
        tmax_arr,
        dtype=np.float32
    )

    lat_deg = np.asarray(
        lat_deg,
        dtype=np.float32
    )

    doy_arr = np.asarray(
        doy_arr,
        dtype=np.float32
    )


    # -----------------------------------------------------------
    # Mean temperature
    # -----------------------------------------------------------

    tmean = (
        tmax_arr + tmin_arr
    ) / 2.0


    # -----------------------------------------------------------
    # Temperature range
    # -----------------------------------------------------------

    temp_range = (
        tmax_arr - tmin_arr
    )

    temp_range = np.maximum(
        temp_range,
        0.0
    )


    # -----------------------------------------------------------
    # Convert latitude to radians
    # -----------------------------------------------------------

    lat_rad = np.deg2rad(
        lat_deg
    )


    # -----------------------------------------------------------
    # Solar geometry
    # -----------------------------------------------------------

    doy2 = doy_arr[:, None]


    dr = (
        1.0
        + 0.033
        * np.cos(
            2.0 * np.pi * doy2 / 365.0
        )
    )


    delta = (
        0.409
        * np.sin(
            2.0 * np.pi * doy2 / 365.0
            - 1.39
        )
    )


    # -----------------------------------------------------------
    # Sunset hour angle
    # -----------------------------------------------------------

    cos_ws = (
        -np.tan(lat_rad[None, :])
        * np.tan(delta)
    )


    cos_ws = np.clip(
        cos_ws,
        -1.0,
        1.0
    )


    ws = np.arccos(
        cos_ws
    )


    # -----------------------------------------------------------
    # Extraterrestrial radiation
    # -----------------------------------------------------------

    Gsc = 0.0820


    Ra = (
        (24.0 * 60.0 / np.pi)
        * Gsc
        * dr
        * (
            ws
            * np.sin(lat_rad[None, :])
            * np.sin(delta)
            +
            np.cos(lat_rad[None, :])
            * np.cos(delta)
            * np.sin(ws)
        )
    )


    # -----------------------------------------------------------
    # Hargreaves-Samani PET
    # -----------------------------------------------------------

    pet = (
        0.0023
        * (tmean + 17.8)
        * np.sqrt(temp_range)
        * Ra
    )


    # -----------------------------------------------------------
    # Preserve NaNs
    # -----------------------------------------------------------

    invalid = (
        ~np.isfinite(tmin_arr)
        |
        ~np.isfinite(tmax_arr)
    )

    pet[invalid] = np.nan


    # -----------------------------------------------------------
    # Remove negative PET
    # -----------------------------------------------------------

    finite_pet = np.isfinite(
        pet
    )

    pet[
        finite_pet
        & (pet < 0)
    ] = 0.0


    return pet.astype(
        np.float32
    )


# ---------------------------------------------------------------
# 8. CHECK HF LOGIN
# ---------------------------------------------------------------

print("\nChecking Hugging Face authentication...")


try:

    api = HfApi()

    user_info = api.whoami()

    print(
        "Logged in to Hugging Face as:",
        user_info.get("name")
    )

except Exception:

    print(
        "Please log in to Hugging Face."
    )

    login()

    api = HfApi()

    user_info = api.whoami()

    print(
        "Logged in as:",
        user_info.get("name")
    )


# ---------------------------------------------------------------
# 9. CHECK HF REPOSITORY
# ---------------------------------------------------------------

print(
    "\nChecking Hugging Face repository..."
)


try:

    repo_files = api.list_repo_files(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE
    )

except Exception as e:

    raise RuntimeError(
        "Could not access Hugging Face repository.\n"
        f"Repository: {HF_REPO}\n"
        f"Error: {repr(e)}"
    )


existing_chunks = set()


for filename in repo_files:

    if not filename.startswith(
        "chunks/chunk_"
    ):

        continue

    if not filename.endswith(
        ".npz"
    ):

        continue


    name = os.path.basename(
        filename
    )

    try:

        number = int(
            name[
                len("chunk_"):
                -len(".npz")
            ]
        )

        existing_chunks.add(
            number
        )

    except Exception:

        pass


print(
    "Existing Daymet chunks on Hugging Face:",
    len(existing_chunks)
)


if existing_chunks:

    print(
        "First existing chunk:",
        min(existing_chunks)
    )

    print(
        "Last existing chunk:",
        max(existing_chunks)
    )


missing_chunks = [
    i
    for i in range(
        1,
        TOTAL_CHUNKS + 1
    )
    if i not in existing_chunks
]


print(
    "Chunks still needed:",
    len(missing_chunks)
)


if missing_chunks:

    print(
        "First missing chunks:",
        missing_chunks[:20]
    )


# ---------------------------------------------------------------
# 10. FRESH DAYMET CONNECTION
# ---------------------------------------------------------------

def open_fresh_daymet():

    """
    Open the official Daymet collection-level Zarr.

    IMPORTANT:
    We use the xarray storage options supplied by the
    Planetary Computer STAC asset directly.

    We do NOT manually construct a SAS token request.
    """

    print(
        "Opening fresh Daymet connection..."
    )


    # -----------------------------------------------------------
    # Clear cached SAS credentials
    # -----------------------------------------------------------

    try:

        pc_sas.TOKEN_CACHE.clear()

        print(
            "Cleared Planetary Computer SAS token cache."
        )

    except Exception as e:

        print(
            "Warning: could not clear token cache:",
            repr(e)
        )


    # -----------------------------------------------------------
    # Open STAC catalog
    # -----------------------------------------------------------

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace
    )


    # -----------------------------------------------------------
    # Daymet collection
    # -----------------------------------------------------------

    collection = catalog.get_collection(
        "daymet-daily-na"
    )


    # -----------------------------------------------------------
    # Official collection-level Zarr
    # -----------------------------------------------------------

    asset = collection.assets[
        "zarr-abfs"
    ]


    print(
        "Daymet Zarr:"
    )

    print(
        asset.href
    )


    # -----------------------------------------------------------
    # Use credentials supplied by STAC
    # -----------------------------------------------------------

    open_kwargs = dict(
        asset.extra_fields[
            "xarray:open_kwargs"
        ]
    )


    storage_options = dict(
        asset.extra_fields[
            "xarray:storage_options"
        ]
    )


    print(
        "Storage account:",
        storage_options.get(
            "account_name"
        )
    )


    credential = storage_options.get(
        "credential"
    )


    if credential is None:

        raise RuntimeError(
            "Planetary Computer did not provide "
            "an ABFS credential."
        )


    print(
        "Credential supplied by "
        "Planetary Computer STAC."
    )


    # -----------------------------------------------------------
    # Open Zarr
    # -----------------------------------------------------------

    print(
        "Opening Daymet Zarr..."
    )


    ds = xr.open_zarr(
        asset.href,
        **open_kwargs,
        storage_options=storage_options
    )


    # -----------------------------------------------------------
    # Select requested period
    # -----------------------------------------------------------

    ds = ds.sel(
        time=slice(
            START,
            END
        )
    )


    print(
        "Daymet opened successfully."
    )


    print(
        "Selected time steps:",
        ds.sizes["time"]
    )


    if ds.sizes["time"] != N_TIME_EXPECTED:

        ds.close()

        raise RuntimeError(
            f"Expected {N_TIME_EXPECTED} "
            f"time steps, got "
            f"{ds.sizes['time']}"
        )


    return ds


# ---------------------------------------------------------------
# 11. VALIDATE CHUNK
# ---------------------------------------------------------------

def validate_chunk(
    filename,
    expected_cells
):

    """
    Validate an NPZ climate chunk.
    """

    if not os.path.exists(
        filename
    ):

        return False


    try:

        with np.load(
            filename,
            allow_pickle=False
        ) as z:

            required = [
                "tmin",
                "tmax",
                "prcp",
                "pet",
            ]


            for var in required:

                if var not in z.files:

                    print(
                        f"Missing variable {var} "
                        f"in {filename}"
                    )

                    return False


                arr = z[var]


                expected_shape = (
                    N_TIME_EXPECTED,
                    expected_cells
                )


                if arr.shape != expected_shape:

                    print(
                        f"Wrong shape for {var}: "
                        f"{arr.shape}, "
                        f"expected {expected_shape}"
                    )

                    return False


                if arr.dtype != np.float32:

                    print(
                        f"Wrong dtype for {var}: "
                        f"{arr.dtype}"
                    )

                    return False


                # Need at least some finite values
                if not np.isfinite(
                    arr
                ).any():

                    print(
                        f"No finite values for {var}"
                    )

                    return False


        return True


    except Exception as e:

        print(
            f"Chunk validation failed: "
            f"{filename}"
        )

        print(
            repr(e)
        )

        return False


# ---------------------------------------------------------------
# 12. PROCESS ONE CHUNK
# ---------------------------------------------------------------

def process_chunk(
    chunk_number,
    start_idx,
    end_idx
):

    """
    Extract one 2000-cell chunk from Daymet,
    calculate PET, save NPZ locally,
    upload to Hugging Face, and verify.
    """

    n = (
        end_idx - start_idx
    )


    print("\n" + "=" * 70)

    print(
        f"PROCESSING CHUNK "
        f"{chunk_number:03d}"
    )

    print(
        f"Cells: "
        f"{start_idx:,} - "
        f"{end_idx - 1:,}"
    )

    print(
        f"Number of cells: {n:,}"
    )

    print("=" * 70)


    # -----------------------------------------------------------
    # Local temporary paths
    # -----------------------------------------------------------

    local_file = os.path.join(
        TEMP_DIR,
        f"chunk_{chunk_number:03d}.npz"
    )


    partial_file = os.path.join(
        TEMP_DIR,
        f"chunk_{chunk_number:03d}.partial.npz"
    )


    # -----------------------------------------------------------
    # If a valid local temporary file exists,
    # use it instead of downloading again.
    # -----------------------------------------------------------

    if validate_chunk(
        local_file,
        n
    ):

        print(
            "Valid local temporary chunk "
            "already exists."
        )

        print(
            "Skipping Daymet extraction."
        )

    else:

        if os.path.exists(
            local_file
        ):

            os.remove(
                local_file
            )


        # -------------------------------------------------------
        # Pixel coordinates
        # -------------------------------------------------------

        yy = y_indices[
            start_idx:end_idx
        ]

        xx = x_indices[
            start_idx:end_idx
        ]


        chunk_lat = lat[
            start_idx:end_idx
        ].astype(
            np.float32
        )


        # -------------------------------------------------------
        # Retry Daymet extraction
        # -------------------------------------------------------

        extraction_success = False


        for attempt in range(
            1,
            MAX_DAYMET_RETRIES + 1
        ):

            ds = None


            try:

                print(
                    f"\nDaymet extraction "
                    f"attempt "
                    f"{attempt}/"
                    f"{MAX_DAYMET_RETRIES}"
                )


                # ------------------------------------------------
                # Open fresh Daymet connection
                # ------------------------------------------------

                ds = open_fresh_daymet()


                # ------------------------------------------------
                # Extract only requested pixels
                # ------------------------------------------------

                print(
                    "Selecting pixel data..."
                )


                tmin_da = ds[
                    "tmin"
                ].isel(
                    y=xr.DataArray(
                        yy,
                        dims="cell"
                    ),
                    x=xr.DataArray(
                        xx,
                        dims="cell"
                    )
                )


                tmax_da = ds[
                    "tmax"
                ].isel(
                    y=xr.DataArray(
                        yy,
                        dims="cell"
                    ),
                    x=xr.DataArray(
                        xx,
                        dims="cell"
                    )
                )


                prcp_da = ds[
                    "prcp"
                ].isel(
                    y=xr.DataArray(
                        yy,
                        dims="cell"
                    ),
                    x=xr.DataArray(
                        xx,
                        dims="cell"
                    )
                )


                print(
                    "Downloading selected "
                    "Daymet values..."
                )


                # ------------------------------------------------
                # Compute only selected pixels
                # ------------------------------------------------

                tmin_arr = (
                    tmin_da
                    .compute()
                    .values
                )


                tmax_arr = (
                    tmax_da
                    .compute()
                    .values
                )


                prcp_arr = (
                    prcp_da
                    .compute()
                    .values
                )


                print(
                    "Daymet data downloaded."
                )


                # ------------------------------------------------
                # Convert masked values to NaN
                # ------------------------------------------------

                tmin_arr = np.asarray(
                    tmin_arr,
                    dtype=np.float32
                )


                tmax_arr = np.asarray(
                    tmax_arr,
                    dtype=np.float32
                )


                prcp_arr = np.asarray(
                    prcp_arr,
                    dtype=np.float32
                )


                # ------------------------------------------------
                # Ensure shape
                # ------------------------------------------------

                expected_shape = (
                    N_TIME_EXPECTED,
                    n
                )


                if (
                    tmin_arr.shape
                    != expected_shape
                ):

                    raise RuntimeError(
                        f"tmin shape "
                        f"{tmin_arr.shape} "
                        f"!= "
                        f"{expected_shape}"
                    )


                if (
                    tmax_arr.shape
                    != expected_shape
                ):

                    raise RuntimeError(
                        f"tmax shape "
                        f"{tmax_arr.shape} "
                        f"!= "
                        f"{expected_shape}"
                    )


                if (
                    prcp_arr.shape
                    != expected_shape
                ):

                    raise RuntimeError(
                        f"prcp shape "
                        f"{prcp_arr.shape} "
                        f"!= "
                        f"{expected_shape}"
                    )


                # ------------------------------------------------
                # Calculate PET
                # ------------------------------------------------

                print(
                    "Calculating "
                    "Hargreaves-Samani PET..."
                )


                pet_arr = (
                    hargreaves_samani_pet(
                        tmin_arr,
                        tmax_arr,
                        chunk_lat,
                        doy
                    )
                )


                # ------------------------------------------------
                # Validate PET
                # ------------------------------------------------

                if pet_arr.shape != expected_shape:

                    raise RuntimeError(
                        f"PET shape "
                        f"{pet_arr.shape} "
                        f"!= "
                        f"{expected_shape}"
                    )


                # ------------------------------------------------
                # Save local NPZ
                # ------------------------------------------------

                print(
                    "Saving local NPZ..."
                )


                if os.path.exists(
                    partial_file
                ):

                    os.remove(
                        partial_file
                    )


                np.savez_compressed(
                    partial_file,
                    tmin=tmin_arr,
                    tmax=tmax_arr,
                    prcp=prcp_arr,
                    pet=pet_arr
                )


                # ------------------------------------------------
                # Validate saved file
                # ------------------------------------------------

                if not validate_chunk(
                    partial_file,
                    n
                ):

                    raise RuntimeError(
                        "Saved NPZ failed validation."
                    )


                # ------------------------------------------------
                # Atomic local rename
                # ------------------------------------------------

                os.replace(
                    partial_file,
                    local_file
                )


                print(
                    "Local chunk saved:"
                )

                print(
                    local_file
                )


                extraction_success = True

                break


            except Exception as e:

                print(
                    "\nDaymet extraction failed:"
                )

                print(
                    repr(e)
                )


                if (
                    ds is not None
                ):

                    try:

                        ds.close()

                    except Exception:

                        pass


                if os.path.exists(
                    partial_file
                ):

                    try:

                        os.remove(
                            partial_file
                        )

                    except Exception:

                        pass


                if attempt < MAX_DAYMET_RETRIES:

                    wait_seconds = 20

                    print(
                        f"Waiting "
                        f"{wait_seconds} "
                        f"seconds before retry..."
                    )

                    time.sleep(
                        wait_seconds
                    )


            finally:

                if (
                    ds is not None
                ):

                    try:

                        ds.close()

                    except Exception:

                        pass

                    del ds


                gc.collect()


        if not extraction_success:

            raise RuntimeError(
                f"Could not extract "
                f"chunk {chunk_number:03d}"
            )


    # -----------------------------------------------------------
    # Upload to Hugging Face
    # -----------------------------------------------------------

    hf_path = (
        f"chunks/"
        f"chunk_{chunk_number:03d}.npz"
    )


    print(
        "\nUploading to Hugging Face:"
    )

    print(
        hf_path
    )


    try:

        api.upload_file(
            path_or_fileobj=local_file,
            path_in_repo=hf_path,
            repo_id=HF_REPO,
            repo_type=HF_REPO_TYPE
        )


        print(
            "Upload completed."
        )


    except Exception as e:

        print(
            "\nHugging Face upload failed:"
        )

        print(
            repr(e)
        )

        print(
            "LOCAL FILE HAS BEEN KEPT:"
        )

        print(
            local_file
        )

        raise


    # -----------------------------------------------------------
    # Verify remote file
    # -----------------------------------------------------------

    print(
        "Verifying remote file..."
    )


    try:

        updated_files = api.list_repo_files(
            repo_id=HF_REPO,
            repo_type=HF_REPO_TYPE
        )


        if hf_path not in updated_files:

            raise RuntimeError(
                "Upload returned successfully, "
                "but the file was not found "
                "in the repository."
            )


    except Exception as e:

        print(
            "\nRemote verification failed:"
        )

        print(
            repr(e)
        )

        print(
            "LOCAL FILE HAS BEEN KEPT:"
        )

        print(
            local_file
        )

        raise


    print(
        f"Chunk {chunk_number:03d} "
        "successfully uploaded and verified."
    )


    # -----------------------------------------------------------
    # Delete local copy ONLY after successful verification
    # -----------------------------------------------------------

    try:

        os.remove(
            local_file
        )

        print(
            "Local temporary file deleted."
        )

    except Exception as e:

        print(
            "Could not delete local file:",
            repr(e)
        )


    # -----------------------------------------------------------
    # Cleanup
    # -----------------------------------------------------------

    del yy
    del xx
    del chunk_lat

    gc.collect()


# ---------------------------------------------------------------
# 13. MAIN EXTRACTION LOOP
# ---------------------------------------------------------------

print("\n")
print("=" * 70)
print("STARTING EXTRACTION")
print("=" * 70)

print(
    f"Total chunks: {TOTAL_CHUNKS}"
)

print(
    f"Already on HF: "
    f"{len(existing_chunks)}"
)

print(
    f"Remaining: "
    f"{len(missing_chunks)}"
)

print("=" * 70)


failed_chunks = []


for chunk_number in range(
    1,
    TOTAL_CHUNKS + 1
):

    # -----------------------------------------------------------
    # Skip chunks already on HF
    # -----------------------------------------------------------

    if chunk_number in existing_chunks:

        print(
            f"\nChunk "
            f"{chunk_number:03d} "
            f"already exists on "
            f"Hugging Face. SKIPPING."
        )

        continue


    # -----------------------------------------------------------
    # Determine cell range
    # -----------------------------------------------------------

    start_idx = (
        (chunk_number - 1)
        * CELL_CHUNK
    )


    end_idx = min(
        start_idx + CELL_CHUNK,
        ncell
    )


    try:

        process_chunk(
            chunk_number,
            start_idx,
            end_idx
        )


        # Add to in-memory set so it will not
        # be processed again during this run.

        existing_chunks.add(
            chunk_number
        )


        print(
            f"\n✓ CHUNK "
            f"{chunk_number:03d} COMPLETE"
        )


    except Exception as e:

        print(
            "\n" + "!" * 70
        )

        print(
            f"CHUNK "
            f"{chunk_number:03d} FAILED"
        )

        print(
            repr(e)
        )

        print(
            "!" * 70
        )


        failed_chunks.append(
            chunk_number
        )


        # Stop here rather than silently skipping
        # a failed chunk.

        print(
            "\nStopping extraction."
        )

        print(
            "The completed chunks are safe."
        )

        print(
            "Rerun this cell to resume."
        )

        raise


# ---------------------------------------------------------------
# 14. FINAL VERIFICATION
# ---------------------------------------------------------------

print("\n")
print("=" * 70)
print("FINAL HUGGING FACE VERIFICATION")
print("=" * 70)


final_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type=HF_REPO_TYPE
)


final_chunks = set()


for filename in final_files:

    if not filename.startswith(
        "chunks/chunk_"
    ):

        continue

    if not filename.endswith(
        ".npz"
    ):

        continue


    name = os.path.basename(
        filename
    )


    try:

        number = int(
            name[
                len("chunk_"):
                -len(".npz")
            ]
        )

        final_chunks.add(
            number
        )

    except Exception:

        pass


expected_chunks = set(
    range(
        1,
        TOTAL_CHUNKS + 1
    )
)


missing_final = (
    expected_chunks
    - final_chunks
)


unexpected_final = (
    final_chunks
    - expected_chunks
)


print(
    f"Total Daymet chunks on HF: "
    f"{len(final_chunks)}"
)


print(
    f"Expected: "
    f"{TOTAL_CHUNKS}"
)


if missing_final:

    print(
        "\nMISSING CHUNKS:"
    )

    print(
        sorted(
            missing_final
        )
    )

else:

    print(
        "\n✓ NO CHUNKS MISSING"
    )


if unexpected_final:

    print(
        "\nUNEXPECTED CHUNKS:"
    )

    print(
        sorted(
            unexpected_final
        )
    )

else:

    print(
        "✓ NO UNEXPECTED CHUNKS"
    )


# ---------------------------------------------------------------
# 15. SUMMARY
# ---------------------------------------------------------------

print("\n")
print("=" * 70)
print("EXTRACTION SUMMARY")
print("=" * 70)


print(
    f"Total cells: "
    f"{ncell:,}"
)

print(
    f"Time steps: "
    f"{N_TIME_EXPECTED:,}"
)

print(
    f"Total chunks: "
    f"{TOTAL_CHUNKS}"
)

print(
    f"Chunks on HF: "
    f"{len(final_chunks)}"
)

print(
    f"Missing chunks: "
    f"{len(missing_final)}"
)


if not missing_final:

    print("\n")
    print(
        "🎉 ALL 242 CHUNKS ARE ON HUGGING FACE!"
    )

    print(
        "The Daymet pixel extraction is complete."
    )

else:

    print("\n")
    print(
        "Extraction is not yet complete."
    )

    print(
        "Missing chunks:",
        sorted(missing_final)
    )


print("=" * 70)